# D1. 학습량 곡선 — 학습 문장을 몇 개까지 넣는 게 효율적인가

**질문**: 학습 문장을 1,500 → 6,000 → 2만 → 6만 → 20만으로 늘리면 점수가 어떻게 오르고, 어디서 멈추는가? split_v1의 6만이 적절한가?

**설계**
- 모델은 기준선 하나(klue/roberta-base)로 고정한다. 데이터 양에 대한 반응은 같은 급의 모델끼리 비슷하다고 보고, 측정 도구로 쓴다.
- 작은 세트가 큰 세트에 **포함**되도록 만들었다(`src/make_experiment_sets.py`). 1,500 ⊂ 6,000 ⊂ … ⊂ 20만. 그래서 점수 변화는 "문장이 늘어난 효과"만 담는다.
- dev·test는 split_v1 그대로. 크기를 고를 때는 **dev 점수**로 판단하고, test는 보고용이다.
- 에폭은 모두 3으로 같게 둔다. 큰 세트일수록 학습 스텝이 많아진다.

**시간**: 20만은 6만의 3.3배 걸린다. 6만 학습 시간을 먼저 재고(M1의 klue_roberta와 같은 실험이라 결과를 재사용한다), 20만은 L4·A100·G4 같은 빠른 GPU에서 돌리길 권한다.

## 0. 공통 준비 (모든 노트북 같음)
Colab → 런타임 → 런타임 유형 변경 → GPU(T4 이상). 구글 드라이브를 연결하면 이 PC의 프로젝트 폴더가 `/content/drive/Othercomputers/내 PC/...` 아래에 보인다. 경로가 다르면 왼쪽 파일 탭에서 찾아 `PROJECT` 만 고친다.

공용 코드 `src/ner_lib.py`(학습)와 `src/evaluate.py`(팀 공용 채점기)를 불러온다. 정렬·복원·채점 규칙이 모든 노트북에서 같아야 비교가 공정하기 때문이다.

**저장되는 것**: 실험마다 `reports/results/{실험}.json`(에폭별 dev 점수 포함), `preds/`(예측), `scores/`(채점), `errors/`(오답), 그리고 `runs.jsonl`(실행 기록 한 줄). 그림은 R0 노트북이 이 파일들로 그린다.
**끊겼을 때**: 같은 셀을 다시 실행하면 끝난 실험은 건너뛰고, 진행 중이던 실험은 마지막 에폭 체크포인트부터 이어서 학습한다.

**최종 가중치(10/23부터 항상 저장)**: 학습이 끝나면 예측·채점보다 먼저 `내 드라이브/ner_models/{실험}/` 에 가중치(model.safetensors 또는 crf_model_state.pt)와 토크나이저를 저장하고, 다시 읽어 학습된 값과 같은지 확인한다. 확인에 실패하면 체크포인트를 지우지 않고 멈춘다. 런타임 반납 전에도 한 번 더 확인한다.

In [ ]:
!pip -q install -U transformers datasets accelerate pytorch-crf kiwipiepy
import sys, os, json, torch
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/Othercomputers/내 PC/NLP/프로젝트1/공통2_개체명인식_NER"
sys.path.insert(0, f"{PROJECT}/src")
import importlib, ner_lib; importlib.reload(ner_lib)
MODELS = "/content/drive/MyDrive/ner_models"   # 최종 가중치(실험마다 폴더). 학습 직후 저장하고 다시 읽어 확인한다. 지우지 말 것
ner_lib.DEFAULTS.update(model_dir=MODELS)
from ner_lib import run_experiment, load_results, results_table, upper_bound, load_jsonl, prepare_aihub_model
SPLIT   = f"{PROJECT}/data/split_v1"          # dev 1만 / test 1만 (모든 버전 공통)
TRAIN   = f"{PROJECT}/data/split_v3/train.jsonl"   # 팀 기준 학습 데이터: 30만 (2026-10-22 결정)
SUFFIX  = "_300k"                              # 30만으로 학습한 실험 이름 꼬리표
RESULTS = f"{PROJECT}/reports/results"        # 결과 json이 모이는 곳 (팀원 모두 같은 폴더)
CKPT    = "/content/drive/MyDrive/ner_ckpt"    # 에폭 체크포인트. 끊겨도 이어서 학습. '내 드라이브'라 PC로 동기화되지 않음
MEMBER  = "sejin_choi"                          # ← 실행한 사람 이름, 영어로 (결과 기록 runs.jsonl에 남음)
ner_lib.DEFAULTS.update(ckpt_dir=CKPT, member=MEMBER)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

In [ ]:
# 한글 글꼴 (그래프 글자가 네모로 깨지지 않게)
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, glob as _g
for f in _g.glob("/usr/share/fonts/truetype/nanum/*.ttf"): fm.fontManager.addfont(f)
plt.rc("font", family="NanumGothic"); plt.rc("axes", unicode_minus=False)
FIG = f"{PROJECT}/reports/figures"; os.makedirs(FIG, exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=150); plt.show(); print("저장:", f"{FIG}/{name}.png")

In [ ]:
ner_lib.DEFAULTS["stage"] = "D1_학습량곡선"
EXP = f"{PROJECT}/data/exp_sets"
BASE = "klue/roberta-base"
SIZES = [1500, 6000, 20000, 60000, 200000]
COMMON = dict(dev=f"{SPLIT}/dev.jsonl", test=f"{SPLIT}/test.jsonl", out_dir=RESULTS, model=BASE,
              max_len=512, lr=5e-5, epochs=3, batch=16, seed=42)
print(json.load(open(f"{EXP}/stats.json", encoding="utf-8"))["overlap_check"])

## 1. 실행
6만은 M1의 `M1_klue_roberta` 와 **완전히 같은 실험**(같은 6만 문장, 같은 순서, 같은 시드)이라, 그 결과가 있으면 복사해서 쓴다. 나머지만 학습한다.

In [ ]:
import shutil
for n in SIZES:
    exp = f"D1_lc_{n}"
    if n == 60000 and os.path.exists(f"{RESULTS}/M1_klue_roberta.json") and not os.path.exists(f"{RESULTS}/{exp}.json"):
        r = json.load(open(f"{RESULTS}/M1_klue_roberta.json", encoding="utf-8")); r["exp"] = exp
        json.dump(r, open(f"{RESULTS}/{exp}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        print("6만: M1 결과 재사용"); continue
    run_experiment({**COMMON, "train": f"{EXP}/lc_train_{n}.jsonl", "exp": exp})

## 2. 곡선 그리기와 판단
x축은 로그 스케일이다(문장 수를 3~4배씩 늘렸으므로). 판단 기준 예: **세트를 3배 늘렸을 때 dev F1이 0.5점 미만 오르면** 그 앞 크기를 효율적인 크기로 본다. 이 기준은 팀이 정하면 된다.

In [ ]:
import matplotlib.pyplot as plt
rs = sorted([r for r in load_results(RESULTS) if r["exp"].startswith("D1_lc_")], key=lambda r: r["n_train"])
n = [r["n_train"] for r in rs]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(n, [r["dev"]["ALL"]["f1"] for r in rs], "o-", label="dev F1 (micro)")
ax[0].plot(n, [r["dev"]["MACRO_F1"] for r in rs], "s--", label="dev macro F1")
ax[0].set_xscale("log"); ax[0].set_xlabel("학습 문장 수"); ax[0].set_ylabel("F1"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(n, [r["train_sec"] / 60 for r in rs], "o-"); ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("학습 문장 수"); ax[1].set_ylabel("학습 시간(분)"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
for a, b in zip(rs, rs[1:]):
    print(f"{a['n_train']:>7,} → {b['n_train']:>7,}: dev F1 {a['dev']['ALL']['f1']:.4f} → {b['dev']['ALL']['f1']:.4f} "
          f"(+{(b['dev']['ALL']['f1']-a['dev']['ALL']['f1'])*100:.2f}점), 시간 ×{b['train_sec']/max(a['train_sec'],1):.1f}")

In [ ]:
# 전체 데이터(약 5,500만 문장) 학습 시간 추정: 실제로 잰 처리 속도로 계산
sps = max(r["train_samples_per_second"] for r in rs)
for ep in (1, 3):
    h = 55_000_000 * ep / sps / 3600
    print(f"이 GPU({rs[-1]['gpu']})에서 전체 {ep}에폭: 약 {h:,.0f}시간 (초당 {sps:,.0f}문장 기준)")

## 3. 태그별로도 보기
전체 F1이 멈춰도 드문 태그(NW·TR·FD)는 계속 오를 수 있다. 그렇다면 "전체를 늘리기"보다 "드문 태그 문장만 더 넣기"가 효율적이라는 근거가 된다.

In [ ]:
import pandas as pd
pd.DataFrame([{**{"n_train": r["n_train"]}, **{t: r["dev"][t]["f1"] for t in ner_lib.TAGS}} for r in rs]).set_index("n_train")

## 가중치 저장 확인
이번에 돌린 실험마다 `내 드라이브/ner_models/{실험}/` 에 가중치가 있는지 확인한다. 없으면 에러로 멈춘다.

In [ ]:
ner_lib.verify_saved_models()   # 이번에 다룬 실험의 최종 가중치가 하나라도 없으면 여기서 멈추고 반납하지 않음
print("가중치 위치:", MODELS)
